# Vector (Semantic) Search and Hybrid Retrieval (RRF)

**DSC 785 Text Analytics, Summer 2026, Week 5 lesson**

**Picks up from:** the previous class, which ended at BM25 (Atlas Search) + Retrieval@10 and a *preview* of hybrid.

The corpus is the course collection of LLM-extracted 10-K/10-Q company analyses in MongoDB Atlas (`TextAnalyticsDB.10K_Analysis`), loaded in Project 2. It is not redistributed here; see the README.

## Lesson plan

1. **Recap (2 min).** Last class we had keyword search (`$text`) and BM25 (`$search`) working, with one
   Atlas search index used (`default`). We scored quality with **Retrieval@10** and saw BM25 surface the
   real quantum pure-plays. We *previewed* hybrid but didn't build it.
2. **Teach embeddings (10 min).** Meaning → vectors; similar meaning = high cosine even with no shared
   words. This is the "dense" / semantic side, vs. BM25's "sparse" / keyword side.
3. **Embed the corpus (10 min).** Generate an embedding for every company's narrative and store it on
   the document (self-healing loop, same pattern as before).
4. **Create the 2nd Atlas index = Vector Search (5 min).** Free cluster allows 3 search indexes; BM25
   was #1, vector is #2. Run `$vectorSearch`.
5. **Compare BM25 vs. vector (10 min).** On "quantum computing," does vector recover the pure-play that
   BM25 missed (e.g., QUBT)?
6. **Hybrid with RRF (10 min).** Fuse BM25 + vector by **rank** (not score), recompute Retrieval@10,
   show hybrid ties or beats both.
7. **Connect to Project 2 (3 min).** This is **Part F** of Project 2 (notebook 04 in this repo). The embedding **bake-off** (Part G) and the
   **RAG** Q&A (Project 2B) come next.

## 0. Reconnect and confirm last week's state

In [ ]:
!pip install pymongo sentence-transformers -q
import pymongo, json, re
import pandas as pd, numpy as np

uri = 'mongodb+srv://<your_user>:<your_password>@<your-cluster>.mongodb.net/'   # your class cluster URI
client = pymongo.MongoClient(uri)
col = client['TextAnalyticsDB']['10K_Analysis']
print('Documents in collection:', col.count_documents({}))     # ~1,754 from last class

# Last week's two keyword searches (return just the tickers, best-first):
def text_search(query, limit=10):
    res = col.find({'$text': {'$search': query}},
                   {'ticker':1,'score':{'$meta':'textScore'},'_id':0}
                  ).sort([('score', {'$meta':'textScore'})]).limit(limit)
    return [d['ticker'] for d in res]

def atlas_search(query, limit=10, path='executive_summary'):       # BM25
    res = col.aggregate([
        {'$search': {'index':'default','text':{'query':query,'path':path}}},
        {'$project': {'ticker':1,'_id':0}}, {'$limit':limit}])
    return [d['ticker'] for d in res]

print('BM25 "quantum computing":', atlas_search('quantum computing'))

## 1. Embeddings — turn *meaning* into vectors

In [ ]:
import sentence_transformers
model = sentence_transformers.SentenceTransformer('all-MiniLM-L6-v2')   # 384-dim, fast

# Intuition: same MEANING -> high cosine, even with ZERO shared words.
a = model.encode('revenue beat analyst expectations', normalize_embeddings=True)
b = model.encode('top-line exceeded consensus',        normalize_embeddings=True)
c = model.encode('the cafeteria served pizza',         normalize_embeddings=True)
print('similar meaning :', round(float(a @ b), 3))   # high  (~0.6+)
print('unrelated       :', round(float(a @ c), 3))   # low
# THIS is what keyword search can't do: 'beat expectations' vs 'exceeded consensus' share no words.

## 2. Embed every company's narrative, store it on the document

In [ ]:
def investable_text(doc):
    parts = [doc.get('executive_summary',''), doc.get('analyst_notes','')]
    for sec in ['strengths','weaknesses','opportunities','threats']:
        parts += [it.get('item','') for it in doc.get('swot',{}).get(sec,[])]
    for r in doc.get('risk_factor_changes',{}).get('new_risks',[]):
        parts.append(r.get('risk','') if isinstance(r,dict) else str(r))
    return ' '.join(p for p in parts if p)

# Self-healing: only embed docs that don't have an embedding yet (re-run safe).
todo = list(col.find({'embedding': {'$exists': False}}))
print('to embed:', len(todo))
for d in todo:
    vec = model.encode(investable_text(d), normalize_embeddings=True).tolist()
    col.update_one({'_id': d['_id']}, {'$set': {'embedding': vec}})
print('done — every document now has a 384-number "meaning" vector')

## 3. Create the Atlas VECTOR SEARCH index (UI step), then query it

In [ ]:
# In Atlas:  your collection -> Atlas Search tab -> Create Search Index -> "Vector Search" -> JSON Editor:
#   { "fields": [ { "type":"vector", "path":"embedding", "numDimensions":384, "similarity":"cosine" } ] }
# Name it:  vector_index
# (This is your 2nd of 3 free search indexes — BM25 'default' was the 1st. Wait for status = Active.)

def vector_search(query, limit=10):
    qv = model.encode(query, normalize_embeddings=True).tolist()
    res = col.aggregate([
        {'$vectorSearch': {'index':'vector_index', 'path':'embedding',
                           'queryVector':qv, 'numCandidates':100, 'limit':limit}},
        {'$project': {'ticker':1,'_id':0}}])
    return [d['ticker'] for d in res]

print('VECTOR "quantum computing":', vector_search('quantum computing'))

## 4. BM25 vs. Vector on the same query

In [ ]:
q = 'quantum computing'
print('keyword $text :', text_search(q))
print('BM25  $search :', atlas_search(q))
print('VECTOR        :', vector_search(q))
# Watch for the pure-plays IONQ / RGTI / QBTS / QUBT. Vector should pull in semantically-quantum
# companies even when they don't repeat the exact phrase 'quantum computing'.

## 5. Quantify it — Retrieval@10 against a hand-built truth set

In [ ]:
truth = {'IONQ', 'RGTI', 'QBTS', 'QUBT'}        # the quantum pure-plays we actually want

def recall_at_k(ranked, truth, k=10):
    return len(truth & set(ranked[:k])) / len(truth)

print('text   @10:', recall_at_k(text_search(q, 20),   truth))
print('BM25   @10:', recall_at_k(atlas_search(q, 20),  truth))
print('vector @10:', recall_at_k(vector_search(q, 20), truth))
# Same metric as last class — now it lets us compare keyword vs. semantic on equal footing.

## 6. Hybrid — fuse BM25 + Vector with Reciprocal Rank Fusion (RRF)

In [ ]:
def rrf(*rank_lists, k=60, limit=10):
    scores = {}
    for ranks in rank_lists:                 # each list is best-first
        for pos, ticker in enumerate(ranks):
            scores[ticker] = scores.get(ticker, 0) + 1.0/(k + pos + 1)
    return sorted(scores, key=scores.get, reverse=True)[:limit]

hybrid = rrf(atlas_search(q, 20), vector_search(q, 20))
print('HYBRID     :', hybrid)
print('hybrid @10 :', recall_at_k(hybrid, truth))
# Key point: RRF combines RANKS, not scores. You can't add a BM25 score to a cosine score
# any more than you can add a P/E ratio to an ROE — so we convert both to ranks and average.
# Free MongoDB has no native hybrid, so we do the fusion here in Python.

## 7. Where this goes next (Project 2B preview)

In [ ]:
# Hybrid retrieval IS the "Retrieve" step of RAG:
#   hybrid_retrieve(question) -> grab the right passages -> put them in a prompt ->
#   an LLM writes a GROUNDED, CITED answer.  That's Project 2B (RAG) — next.

## Notes

- If `$vectorSearch` errors with "index not found," the **vector_index isn't Active yet** — refresh the Atlas Search tab until it finishes building.
- The embedding model used here (`all-MiniLM-L6-v2`) **truncates at ~350 words** — a perfect lead-in to the **Part G bake-off** (MiniLM vs. nomic's 8k-token window) for the project.